# EdgeGuard semantic compatibility-only gate
Synthetic CUDA compatibility evidence only. This notebook does not mount Drive, access datasets, stage data, or start training.

In [ ]:
import json
import re
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY = "https://github.com/emrealmaoglu/edgeguard-road.git"
EDGEGUARD_EXPECTED_COMMIT = "REPLACE_WITH_REVIEWED_LOCAL_FIRST_COMMIT_SHA"
PROJECT_ROOT = Path("/content/edgeguard-road")
if not re.fullmatch(r"[0-9a-f]{40}", EDGEGUARD_EXPECTED_COMMIT):
    raise ValueError("Enter the exact reviewed local-first commit SHA")
if PROJECT_ROOT.exists():
    raise FileExistsError("Use a fresh runtime or remove only the old project checkout")
subprocess.run(
    ["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY, str(PROJECT_ROOT)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "checkout", "--detach", EDGEGUARD_EXPECTED_COMMIT],
    check=True,
)
dirty = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain=v1"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if dirty:
    raise RuntimeError("Compatibility checkout must be clean")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PROJECT_ROOT)], check=True)

In [ ]:
COMPAT = Path("/content/edgeguard-compatibility")
LOGS = Path("/content/edgeguard-logs")
command = [
    sys.executable,
    str(PROJECT_ROOT / "scripts/train/install_semantic_stack.py"),
    "--config",
    str(PROJECT_ROOT / "configs/training/segmentation/framework_mmseg.yaml"),
    "--project-root",
    str(PROJECT_ROOT),
    "--project-commit",
    EDGEGUARD_EXPECTED_COMMIT,
    "--config-root",
    str(PROJECT_ROOT / "configs/training/segmentation"),
    "--runtime-current-root",
    "/content/edgeguard-runtime-current",
    "--runtime-py311-root",
    "/content/edgeguard-runtime-py311",
    "--checkout-root",
    "/content/edgeguard-mmseg",
    "--evidence-root",
    str(COMPAT),
    "--log-root",
    str(LOGS),
    "--cache-root",
    "/content/edgeguard-cache",
    "--data-root",
    "/content/edgeguard-synthetic-data",
    "--execute",
]
try:
    subprocess.run(command, check=True)
except subprocess.CalledProcessError:
    import torch

    print(
        json.dumps(
            {
                "python": sys.version,
                "torch": torch.__version__,
                "cuda": torch.version.cuda,
                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                "disk": shutil.disk_usage("/content")._asdict(),
            },
            indent=2,
            default=str,
        )
    )
    for diagnostic in ("bootstrap_failure.json", "compatibility_failures.json", "run_status.json"):
        path = COMPAT / diagnostic
        if path.is_file():
            print(f"=== {diagnostic} ===")
            print(path.read_text(encoding="utf-8"))
    for log in sorted(LOGS.rglob("*.stderr.log")):
        lines = log.read_text(encoding="utf-8", errors="replace").splitlines()
        if lines:
            print(f"=== stderr tail: {log.name} ===")
            print("\n".join(lines[-120:]))
    for log in sorted(LOGS.rglob("*.stdout.log")):
        lines = log.read_text(encoding="utf-8", errors="replace").splitlines()
        tail = lines[-120:]
        if any(
            any(token in line.lower() for token in ("error", "failed", "traceback"))
            for line in tail
        ):
            print(f"=== error-bearing stdout tail: {log.name} ===")
            print("\n".join(tail))
    raise

In [ ]:
receipt = json.loads((COMPAT / "compatibility_receipt.json").read_text(encoding="utf-8"))
if receipt["project_commit"] != EDGEGUARD_EXPECTED_COMMIT:
    raise RuntimeError("Compatibility receipt belongs to another project commit")
if receipt["five_model_probe"]["model_count"] != 5:
    raise RuntimeError("Five-model compatibility probe is incomplete")
if receipt["five_model_probe"]["checkpoint_resume_verified"] is not True:
    raise RuntimeError("Checkpoint round-trip was not verified")
print(
    json.dumps(
        {
            "status": "compatibility_only_gate_passed",
            "selected_path": receipt["selected_path"],
            "evidence_package": receipt["evidence_package"],
        },
        indent=2,
    )
)
print("STOP: do not mount Drive or start real-data smoke in this notebook.")